# DriftGuard — Dataset Cleaning and Validation

This notebook converts the weakly labelled configuration-drift dataset into
clean, leakage-safe datasets for machine-learning experiments.

It performs:

- Schema and datatype validation
- Invalid-record removal
- Duplicate Diff ID removal
- Within-split duplicate-signature removal
- Cross-split leakage removal
- High-confidence training selection
- Controlled provisional low-risk sampling
- Benign-class downsampling
- Sample-weight generation
- Commit-disjoint temporal train/dev partitioning
- Validation and test isolation
- Clean dataset export

Validation and test repository records are never used for model fitting.

In [1]:
import os
import sys
import json
import math
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd


print("=" * 72)
print("DRIFTGUARD — DATASET CLEANING AND VALIDATION")
print("=" * 72)

current_directory = Path.cwd().resolve()

if current_directory.name.lower() == "notebooks":
    PROJECT_ROOT = current_directory.parent
else:
    PROJECT_ROOT = current_directory

CONFIGS_DIR = PROJECT_ROOT / "configs"
LABELS_DIR = PROJECT_ROOT / "data" / "labels"
CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "clean"
EVALUATION_DIR = PROJECT_ROOT / "data" / "evaluation"
LOGS_DIR = PROJECT_ROOT / "logs"

CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print("Python version   :", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Conda environment:", os.environ.get("CONDA_DEFAULT_ENV", "Not detected"))
print("Project root     :", PROJECT_ROOT)
print("Labels directory :", LABELS_DIR)
print("Clean directory  :", CLEAN_DATA_DIR)

DRIFTGUARD — DATASET CLEANING AND VALIDATION
Python version   : 3.11.15
Python executable: C:\Users\Lenovo\anaconda3\envs\driftguard\python.exe
Conda environment: driftguard
Project root     : C:\Users\Lenovo\Desktop\DriftGuard
Labels directory : C:\Users\Lenovo\Desktop\DriftGuard\data\labels
Clean directory  : C:\Users\Lenovo\Desktop\DriftGuard\data\clean


In [2]:
CLEANING_SETTINGS = {
    "random_seed": 42,

    "valid_splits": [
        "train",
        "validation",
        "test",
    ],

    "valid_labels": [
        "benign",
        "low",
        "medium",
        "high",
        "critical",
        "unlabeled",
    ],

    "valid_operations": [
        "added",
        "deleted",
        "modified",
    ],

    "valid_parser_modes": [
        "structured",
        "line_fallback",
    ],

    # Strong labels included directly in supervised training.
    "minimum_core_training_confidence": 0.85,

    # Lower-confidence catch-all low-risk records.
    "minimum_provisional_low_confidence": 0.70,

    # Number of provisional low-risk records relative to the
    # medium + high + critical training records.
    "provisional_low_to_strong_risk_ratio": 2.0,

    # Maximum benign records relative to all risky records in
    # the final balanced training dataset.
    "benign_to_risky_ratio": 2.0,

    # Final 20% of commits inside each training repository becomes
    # the internal temporal development partition.
    "temporal_development_fraction": 0.20,

    # Preserve test signatures first, then validation, then train.
    "split_retention_priority": {
        "train": 1,
        "validation": 2,
        "test": 3,
    },

    "minimum_sample_weight": 0.10,
    "maximum_sample_weight": 10.00,
}

cleaning_settings_path = (
    CONFIGS_DIR / "dataset_cleaning_settings.json"
)

with cleaning_settings_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        CLEANING_SETTINGS,
        file,
        indent=2,
    )

print(json.dumps(CLEANING_SETTINGS, indent=2))
print("\nSaved settings to:", cleaning_settings_path)

{
  "random_seed": 42,
  "valid_splits": [
    "train",
    "validation",
    "test"
  ],
  "valid_labels": [
    "benign",
    "low",
    "medium",
    "high",
    "critical",
    "unlabeled"
  ],
  "valid_operations": [
    "added",
    "deleted",
    "modified"
  ],
  "valid_parser_modes": [
    "structured",
    "line_fallback"
  ],
  "minimum_core_training_confidence": 0.85,
  "minimum_provisional_low_confidence": 0.7,
  "provisional_low_to_strong_risk_ratio": 2.0,
  "benign_to_risky_ratio": 2.0,
  "temporal_development_fraction": 0.2,
  "split_retention_priority": {
    "train": 1,
    "validation": 2,
    "test": 3
  },
  "minimum_sample_weight": 0.1,
  "maximum_sample_weight": 10.0
}

Saved settings to: C:\Users\Lenovo\Desktop\DriftGuard\configs\dataset_cleaning_settings.json


In [3]:
weak_label_manifest_path = (
    LABELS_DIR / "weak_label_manifest.csv.gz"
)

if not weak_label_manifest_path.exists():
    raise FileNotFoundError(
        "Notebook 04 output is missing:\n"
        f"{weak_label_manifest_path}"
    )

raw_manifest = pd.read_csv(
    weak_label_manifest_path,
    compression="gzip",
    low_memory=False,
)

print("Manifest loaded successfully.")
print("Rows   :", f"{len(raw_manifest):,}")
print("Columns:", len(raw_manifest.columns))

display(raw_manifest.head())

Manifest loaded successfully.
Rows   : 359,738
Columns: 29


,diff_id,source_record_id,repository,dataset_split,commit_hash,commit_author_date,commit_message,file_path,configuration_type,parser_name,...,primary_rule_id,matched_rule_ids,matched_rule_count,rule_conflict,weak_label_reason,label_source,label_usage,sensitive_terms,sensitive_term_count,needs_manual_review
0,c820a8520734fc4ecbbbcd59610480fca6eb145bfbf626...,f3e711748b8342bed50d50afa170591f3dd0cdd2df05b0...,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,helm-chart/Chart.yaml,helm,yaml,...,NaN,[],0,False,No security-sensitive field or dangerous value...,benign_heuristic,weak_training_candidate,[],0,True
1,2778bac3137cbec71eb3ced76ea3d38addea260fd6a570...,f3e711748b8342bed50d50afa170591f3dd0cdd2df05b0...,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,helm-chart/Chart.yaml,helm,yaml,...,NaN,[],0,False,The change modifies a common non-security meta...,benign_heuristic,weak_training_candidate,[],0,False
2,dd772049c2639bec5386ebd1d70965302edda15fd2585f...,4ada560261335ebf7d43ecb68fcd56a2f0f3d380286a6b...,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,helm-chart/values.yaml,helm,yaml,...,NaN,[],0,False,The change modifies a common non-security meta...,benign_heuristic,weak_training_candidate,[],0,False
3,37a016215756d4ed90964a7487ef10b3ed80d5a4d3e8f2...,3cb9882df2d988155d0ea9c634473aa2bb22ba373bad1b...,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,kustomize/base/adservice.yaml,kustomize,yaml,...,NaN,[],0,False,The change modifies a common non-security meta...,benign_heuristic,weak_training_candidate,[],0,False
4,de79a27b115b18a990fedc06348d12fe1e6318c8fe5f7c...,c213b0bf8781befc5354bd284fa14c4cee54b360bb75a5...,microservices_demo,train,9a4616e77f0f9cbcbecaf27d711c38890dda1404,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,kustomize/base/cartservice.yaml,kustomize,yaml,...,NaN,[],0,False,The change modifies a common non-security meta...,benign_heuristic,weak_training_candidate,[],0,False


In [4]:
REQUIRED_COLUMNS = [
    "diff_id",
    "source_record_id",
    "repository",
    "dataset_split",
    "commit_hash",
    "commit_author_date",
    "commit_message",
    "file_path",
    "configuration_type",
    "parser_name",
    "parser_mode",
    "operation",
    "field_path",
    "old_value",
    "new_value",
    "change_signature_sha256",
    "weak_label",
    "weak_label_confidence",
    "severity_rank",
    "primary_rule_id",
    "matched_rule_ids",
    "matched_rule_count",
    "rule_conflict",
    "weak_label_reason",
    "label_source",
    "label_usage",
    "sensitive_terms",
    "sensitive_term_count",
    "needs_manual_review",
]

missing_columns = sorted(
    set(REQUIRED_COLUMNS)
    - set(raw_manifest.columns)
)

if missing_columns:
    raise ValueError(
        "The weak-label manifest is missing columns:\n"
        + "\n".join(missing_columns)
    )

print("Required schema validation: PASSED")
print("Required columns:", len(REQUIRED_COLUMNS))

Required schema validation: PASSED
Required columns: 29


In [5]:
manifest = raw_manifest.copy()


def normalize_boolean(value):
    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    normalized = str(value).strip().lower()

    return normalized in {
        "true",
        "1",
        "yes",
        "y",
        "t",
    }


string_columns = [
    "diff_id",
    "source_record_id",
    "repository",
    "dataset_split",
    "commit_hash",
    "file_path",
    "configuration_type",
    "parser_name",
    "parser_mode",
    "operation",
    "field_path",
    "change_signature_sha256",
    "weak_label",
    "primary_rule_id",
    "label_source",
    "label_usage",
]

for column in string_columns:
    manifest[column] = (
        manifest[column]
        .astype("string")
        .str.strip()
    )

lowercase_columns = [
    "dataset_split",
    "parser_mode",
    "operation",
    "weak_label",
    "label_source",
    "label_usage",
]

for column in lowercase_columns:
    manifest[column] = (
        manifest[column]
        .str.lower()
    )

manifest["weak_label_confidence"] = pd.to_numeric(
    manifest["weak_label_confidence"],
    errors="coerce",
)

manifest["severity_rank"] = pd.to_numeric(
    manifest["severity_rank"],
    errors="coerce",
)

manifest["matched_rule_count"] = pd.to_numeric(
    manifest["matched_rule_count"],
    errors="coerce",
)

manifest["sensitive_term_count"] = pd.to_numeric(
    manifest["sensitive_term_count"],
    errors="coerce",
)

manifest["rule_conflict"] = (
    manifest["rule_conflict"]
    .apply(normalize_boolean)
)

manifest["needs_manual_review"] = (
    manifest["needs_manual_review"]
    .apply(normalize_boolean)
)

manifest["commit_author_datetime"] = pd.to_datetime(
    manifest["commit_author_date"],
    errors="coerce",
    utc=True,
)

print("Datatype normalization completed.")

display(
    manifest[
        [
            "dataset_split",
            "weak_label",
            "weak_label_confidence",
            "rule_conflict",
            "needs_manual_review",
            "commit_author_datetime",
        ]
    ].head()
)

Datatype normalization completed.


,dataset_split,weak_label,weak_label_confidence,rule_conflict,needs_manual_review,commit_author_datetime
0,train,benign,0.72,False,True,2026-07-13 14:40:02+00:00
1,train,benign,0.92,False,False,2026-07-13 14:40:02+00:00
2,train,benign,0.92,False,False,2026-07-13 14:40:02+00:00
3,train,benign,0.92,False,False,2026-07-13 14:40:02+00:00
4,train,benign,0.92,False,False,2026-07-13 14:40:02+00:00


In [6]:
ESSENTIAL_NON_NULL_COLUMNS = [
    "diff_id",
    "source_record_id",
    "repository",
    "dataset_split",
    "commit_hash",
    "file_path",
    "configuration_type",
    "parser_mode",
    "operation",
    "field_path",
    "change_signature_sha256",
    "weak_label",
]

quality_flags = pd.DataFrame(
    index=manifest.index
)

quality_flags["missing_essential_value"] = (
    manifest[ESSENTIAL_NON_NULL_COLUMNS]
    .isna()
    .any(axis=1)
)

quality_flags["invalid_split"] = (
    ~manifest["dataset_split"].isin(
        CLEANING_SETTINGS["valid_splits"]
    )
)

quality_flags["invalid_label"] = (
    ~manifest["weak_label"].isin(
        CLEANING_SETTINGS["valid_labels"]
    )
)

quality_flags["invalid_operation"] = (
    ~manifest["operation"].isin(
        CLEANING_SETTINGS["valid_operations"]
    )
)

quality_flags["invalid_parser_mode"] = (
    ~manifest["parser_mode"].isin(
        CLEANING_SETTINGS["valid_parser_modes"]
    )
)

quality_flags["invalid_confidence"] = (
    manifest["weak_label_confidence"].isna()
    |
    ~manifest["weak_label_confidence"].between(
        0,
        1,
        inclusive="both",
    )
)

quality_flags["empty_change"] = (
    manifest["old_value"].isna()
    & manifest["new_value"].isna()
)

normalized_old_values = (
    manifest["old_value"]
    .fillna("<MISSING>")
    .astype(str)
    .str.strip()
)

normalized_new_values = (
    manifest["new_value"]
    .fillna("<MISSING>")
    .astype(str)
    .str.strip()
)

quality_flags["unchanged_modified_value"] = (
    manifest["operation"].eq("modified")
    & normalized_old_values.eq(
        normalized_new_values
    )
)

quality_flags["hard_invalid"] = (
    quality_flags[
        [
            "missing_essential_value",
            "invalid_split",
            "invalid_label",
            "invalid_operation",
            "invalid_parser_mode",
            "invalid_confidence",
            "empty_change",
            "unchanged_modified_value",
        ]
    ]
    .any(axis=1)
)

quality_report = pd.DataFrame(
    {
        "quality_issue": quality_flags.columns,
        "records": [
            int(quality_flags[column].sum())
            for column in quality_flags.columns
        ],
    }
)

display(quality_report)

print(
    "Hard-invalid records:",
    f"{quality_flags['hard_invalid'].sum():,}",
)

,quality_issue,records
0,missing_essential_value,0
1,invalid_split,0
2,invalid_label,0
3,invalid_operation,0
4,invalid_parser_mode,0
5,invalid_confidence,0
6,empty_change,8536
7,unchanged_modified_value,305
8,hard_invalid,8841


Hard-invalid records: 8,841


In [7]:
rejected_invalid_rows = (
    manifest.loc[
        quality_flags["hard_invalid"]
    ]
    .copy()
)

valid_manifest = (
    manifest.loc[
        ~quality_flags["hard_invalid"]
    ]
    .copy()
)

print("Original records :", f"{len(manifest):,}")
print("Rejected records :", f"{len(rejected_invalid_rows):,}")
print("Remaining records:", f"{len(valid_manifest):,}")

if valid_manifest.empty:
    raise ValueError(
        "No valid rows remain after quality filtering."
    )

Original records : 359,738
Rejected records : 8,841
Remaining records: 350,897


In [8]:
valid_manifest["_structured_priority"] = (
    valid_manifest["parser_mode"]
    .eq("structured")
    .astype(int)
)

valid_manifest["_no_conflict_priority"] = (
    ~valid_manifest["rule_conflict"]
).astype(int)

valid_manifest = valid_manifest.sort_values(
    [
        "diff_id",
        "weak_label_confidence",
        "_structured_priority",
        "_no_conflict_priority",
        "commit_author_datetime",
    ],
    ascending=[
        True,
        False,
        False,
        False,
        True,
    ],
    na_position="last",
)

exact_duplicate_mask = (
    valid_manifest.duplicated(
        subset=["diff_id"],
        keep="first",
    )
)

removed_duplicate_diff_ids = (
    valid_manifest.loc[
        exact_duplicate_mask
    ]
    .copy()
)

valid_manifest = (
    valid_manifest.loc[
        ~exact_duplicate_mask
    ]
    .copy()
)

print(
    "Duplicate Diff ID rows removed:",
    f"{len(removed_duplicate_diff_ids):,}",
)

print(
    "Unique Diff IDs remaining      :",
    f"{valid_manifest['diff_id'].nunique():,}",
)

Duplicate Diff ID rows removed: 0
Unique Diff IDs remaining      : 350,897


In [9]:
valid_manifest = valid_manifest.sort_values(
    [
        "dataset_split",
        "change_signature_sha256",
        "weak_label_confidence",
        "_structured_priority",
        "_no_conflict_priority",
        "commit_author_datetime",
    ],
    ascending=[
        True,
        True,
        False,
        False,
        False,
        True,
    ],
    na_position="last",
)

within_split_duplicate_mask = (
    valid_manifest.duplicated(
        subset=[
            "dataset_split",
            "change_signature_sha256",
        ],
        keep="first",
    )
)

removed_within_split_duplicates = (
    valid_manifest.loc[
        within_split_duplicate_mask
    ]
    .copy()
)

valid_manifest = (
    valid_manifest.loc[
        ~within_split_duplicate_mask
    ]
    .copy()
)

print(
    "Within-split duplicate signatures removed:",
    f"{len(removed_within_split_duplicates):,}",
)

print(
    "Records remaining:",
    f"{len(valid_manifest):,}",
)

Within-split duplicate signatures removed: 129,838
Records remaining: 221,059


In [10]:
signature_split_counts = (
    valid_manifest
    .groupby("change_signature_sha256")[
        "dataset_split"
    ]
    .nunique()
)

cross_split_signatures = set(
    signature_split_counts[
        signature_split_counts > 1
    ].index
)

cross_split_overlap_rows = (
    valid_manifest[
        valid_manifest[
            "change_signature_sha256"
        ].isin(cross_split_signatures)
    ]
    .copy()
)

print(
    "Signatures appearing in multiple splits:",
    f"{len(cross_split_signatures):,}",
)

print(
    "Rows associated with those signatures:",
    f"{len(cross_split_overlap_rows):,}",
)

if not cross_split_overlap_rows.empty:
    overlap_matrix = pd.crosstab(
        cross_split_overlap_rows[
            "change_signature_sha256"
        ],
        cross_split_overlap_rows[
            "dataset_split"
        ],
    )

    overlap_pattern_summary = (
        overlap_matrix
        .gt(0)
        .astype(int)
        .value_counts()
        .reset_index(name="signature_count")
    )

    display(overlap_pattern_summary.head(10))

Signatures appearing in multiple splits: 1,132
Rows associated with those signatures: 2,323


,test,train,validation,signature_count
0,1,1,0,894
1,0,1,1,122
2,1,1,1,59
3,1,0,1,57


In [11]:
split_priority = CLEANING_SETTINGS[
    "split_retention_priority"
]

valid_manifest["_split_priority"] = (
    valid_manifest["dataset_split"]
    .map(split_priority)
)

highest_signature_priority = (
    valid_manifest
    .groupby(
        "change_signature_sha256"
    )["_split_priority"]
    .transform("max")
)

cross_split_removal_mask = (
    valid_manifest["_split_priority"]
    < highest_signature_priority
)

removed_cross_split_rows = (
    valid_manifest.loc[
        cross_split_removal_mask
    ]
    .copy()
)

clean_manifest = (
    valid_manifest.loc[
        ~cross_split_removal_mask
    ]
    .copy()
)

print(
    "Cross-split leakage rows removed:",
    f"{len(removed_cross_split_rows):,}",
)

print(
    "Clean records remaining:",
    f"{len(clean_manifest):,}",
)

print("\nRemoved rows by split:")

display(
    removed_cross_split_rows[
        "dataset_split"
    ]
    .value_counts()
    .rename_axis("dataset_split")
    .reset_index(name="removed_records")
)

Cross-split leakage rows removed: 1,191
Clean records remaining: 219,868

Removed rows by split:


,dataset_split,removed_records
0,train,1075
1,validation,116


In [12]:
repository_split_integrity = (
    clean_manifest
    .groupby("repository")[
        "dataset_split"
    ]
    .agg(
        record_count="count",
        unique_splits="nunique",
        assigned_split="first",
    )
    .reset_index()
)

display(repository_split_integrity)

repositories_in_multiple_splits = (
    repository_split_integrity[
        repository_split_integrity[
            "unique_splits"
        ] > 1
    ]
)

if not repositories_in_multiple_splits.empty:
    raise ValueError(
        "Repository leakage detected across splits."
    )

print("Repository-disjoint split check: PASSED")

,repository,record_count,unique_splits,assigned_split
0,ansible_examples,6139,1,test
1,kube_prometheus,76315,1,train
2,kubernetes_examples,7628,1,validation
3,microservices_demo,57045,1,train
4,terraform_aws_vpc,10131,1,train
5,terraform_eks_blueprints,62610,1,test


Repository-disjoint split check: PASSED


In [13]:
original_split_counts = (
    manifest["dataset_split"]
    .value_counts()
    .rename("original_records")
)

clean_split_counts = (
    clean_manifest["dataset_split"]
    .value_counts()
    .rename("clean_records")
)

split_cleaning_summary = pd.concat(
    [
        original_split_counts,
        clean_split_counts,
    ],
    axis=1,
).fillna(0)

split_cleaning_summary[
    "removed_records"
] = (
    split_cleaning_summary[
        "original_records"
    ]
    - split_cleaning_summary[
        "clean_records"
    ]
)

split_cleaning_summary[
    "retention_percentage"
] = (
    split_cleaning_summary[
        "clean_records"
    ]
    / split_cleaning_summary[
        "original_records"
    ].replace(0, np.nan)
    * 100
)

split_cleaning_summary = (
    split_cleaning_summary
    .reset_index()
    .rename(
        columns={
            "index": "dataset_split",
        }
    )
)

display(split_cleaning_summary)

,dataset_split,original_records,clean_records,removed_records,retention_percentage
0,train,228868,143491,85377,62.695964
1,test,118016,68749,49267,58.253966
2,validation,12854,7628,5226,59.343395


In [14]:
clean_label_distribution = (
    clean_manifest
    .groupby(
        [
            "dataset_split",
            "weak_label",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "records",
        }
    )
)

clean_label_distribution[
    "percentage_within_split"
] = (
    clean_label_distribution["records"]
    / clean_label_distribution.groupby(
        "dataset_split"
    )["records"].transform("sum")
    * 100
)

display(
    clean_label_distribution.sort_values(
        [
            "dataset_split",
            "records",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

,dataset_split,weak_label,records,percentage_within_split
0,test,benign,46499,67.635893
3,test,low,14816,21.550859
5,test,unlabeled,6850,9.963781
1,test,critical,320,0.465461
2,test,high,180,0.261822
4,test,medium,84,0.122184
6,train,benign,70361,49.035131
9,train,low,52965,36.911723
11,train,unlabeled,19101,13.311636
8,train,high,566,0.394450


In [15]:
CORE_TRAINING_LABELS = [
    "benign",
    "medium",
    "high",
    "critical",
]

core_training_mask = (
    clean_manifest[
        "dataset_split"
    ].eq("train")
    &
    clean_manifest[
        "weak_label"
    ].isin(CORE_TRAINING_LABELS)
    &
    clean_manifest[
        "weak_label_confidence"
    ].ge(
        CLEANING_SETTINGS[
            "minimum_core_training_confidence"
        ]
    )
)

core_training_pool = (
    clean_manifest.loc[
        core_training_mask
    ]
    .copy()
)

core_training_pool[
    "training_label_source"
] = "high_confidence_core"

core_training_pool[
    "training_target"
] = core_training_pool[
    "weak_label"
]

print(
    "High-confidence core records:",
    f"{len(core_training_pool):,}",
)

display(
    core_training_pool[
        "training_target"
    ]
    .value_counts()
    .rename_axis("training_target")
    .reset_index(name="records")
)

High-confidence core records: 27,171


,training_target,records
0,benign,26107
1,high,566
2,critical,358
3,medium,140


In [16]:
provisional_low_mask = (
    clean_manifest[
        "dataset_split"
    ].eq("train")
    &
    clean_manifest[
        "weak_label"
    ].eq("low")
    &
    clean_manifest[
        "weak_label_confidence"
    ].ge(
        CLEANING_SETTINGS[
            "minimum_provisional_low_confidence"
        ]
    )
    &
    clean_manifest[
        "parser_mode"
    ].eq("structured")
    &
    ~clean_manifest[
        "rule_conflict"
    ]
    &
    clean_manifest[
        "primary_rule_id"
    ].eq(
        "UNCLASSIFIED_SENSITIVE_FIELD_CHANGE"
    )
)

provisional_low_candidates = (
    clean_manifest.loc[
        provisional_low_mask
    ]
    .copy()
)

strong_risk_count = int(
    core_training_pool[
        "training_target"
    ]
    .isin(
        [
            "medium",
            "high",
            "critical",
        ]
    )
    .sum()
)

requested_provisional_low_count = max(
    1_000,
    int(
        round(
            strong_risk_count
            * CLEANING_SETTINGS[
                "provisional_low_to_strong_risk_ratio"
            ]
        )
    ),
)

provisional_low_target = min(
    len(provisional_low_candidates),
    requested_provisional_low_count,
)

print(
    "Available provisional low-risk candidates:",
    f"{len(provisional_low_candidates):,}",
)

print(
    "Strong medium/high/critical records       :",
    f"{strong_risk_count:,}",
)

print(
    "Provisional low-risk sampling target      :",
    f"{provisional_low_target:,}",
)

Available provisional low-risk candidates: 46,851
Strong medium/high/critical records       : 1,064
Provisional low-risk sampling target      : 2,128


In [17]:
def stratified_round_robin_sample(
    dataframe,
    target_size,
    stratification_columns,
    random_seed,
):
    """
    Select a deterministic, diverse sample across multiple strata.

    Lower-frequency strata receive representation before the same
    stratum contributes many additional records.
    """

    if dataframe.empty or target_size <= 0:
        return dataframe.head(0).copy()

    if len(dataframe) <= target_size:
        return dataframe.copy()

    sampled_data = dataframe.copy()

    stratum_values = (
        sampled_data[
            stratification_columns
        ]
        .fillna("<MISSING>")
        .astype(str)
    )

    sampled_data["_sampling_stratum"] = (
        stratum_values
        .agg("|".join, axis=1)
    )

    random_generator = np.random.default_rng(
        random_seed
    )

    sampled_data["_sampling_random"] = (
        random_generator.random(
            len(sampled_data)
        )
    )

    sampled_data = sampled_data.sort_values(
        [
            "_sampling_stratum",
            "_sampling_random",
        ]
    )

    sampled_data["_stratum_rank"] = (
        sampled_data
        .groupby(
            "_sampling_stratum"
        )
        .cumcount()
    )

    sampled_data = sampled_data.sort_values(
        [
            "_stratum_rank",
            "_sampling_random",
        ]
    )

    sampled_data = sampled_data.head(
        target_size
    )

    return sampled_data.drop(
        columns=[
            "_sampling_stratum",
            "_sampling_random",
            "_stratum_rank",
        ],
        errors="ignore",
    )

In [18]:
provisional_low_sample = (
    stratified_round_robin_sample(
        dataframe=provisional_low_candidates,
        target_size=provisional_low_target,
        stratification_columns=[
            "repository",
            "configuration_type",
            "operation",
        ],
        random_seed=CLEANING_SETTINGS[
            "random_seed"
        ],
    )
)

provisional_low_sample[
    "training_label_source"
] = "provisional_low"

provisional_low_sample[
    "training_target"
] = "low"

print(
    "Provisional low-risk records selected:",
    f"{len(provisional_low_sample):,}",
)

display(
    provisional_low_sample[
        [
            "repository",
            "configuration_type",
            "operation",
        ]
    ]
    .value_counts()
    .reset_index(name="records")
    .head(20)
)

Provisional low-risk records selected: 2,128


,repository,configuration_type,operation,records
0,microservices_demo,json,modified,72
1,microservices_demo,json,added,71
2,kube_prometheus,yaml,modified,71
3,kube_prometheus,yaml,added,71
4,microservices_demo,kubernetes,added,71
5,kube_prometheus,yaml,deleted,71
6,microservices_demo,kubernetes,deleted,71
7,terraform_aws_vpc,terraform,deleted,71
8,microservices_demo,json,deleted,71
9,kube_prometheus,kustomize,modified,71


In [19]:
full_supervised_training_pool = pd.concat(
    [
        core_training_pool,
        provisional_low_sample,
    ],
    ignore_index=True,
)

full_supervised_training_pool = (
    full_supervised_training_pool
    .drop_duplicates(
        subset=["diff_id"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "Full supervised training pool:",
    f"{len(full_supervised_training_pool):,}",
)

full_training_distribution = (
    full_supervised_training_pool[
        "training_target"
    ]
    .value_counts()
    .reindex(
        [
            "benign",
            "low",
            "medium",
            "high",
            "critical",
        ],
        fill_value=0,
    )
    .rename_axis("training_target")
    .reset_index(name="records")
)

display(full_training_distribution)

Full supervised training pool: 29,299


,training_target,records
0,benign,26107
1,low,2128
2,medium,140
3,high,566
4,critical,358


In [20]:
class_counts = (
    full_supervised_training_pool[
        "training_target"
    ]
    .value_counts()
)

number_of_training_records = len(
    full_supervised_training_pool
)

number_of_classes = len(class_counts)

raw_class_weights = {
    class_name: (
        number_of_training_records
        / (
            number_of_classes
            * class_count
        )
    )
    for class_name, class_count
    in class_counts.items()
}

class_weights = {
    class_name: float(
        np.clip(
            weight,
            CLEANING_SETTINGS[
                "minimum_sample_weight"
            ],
            CLEANING_SETTINGS[
                "maximum_sample_weight"
            ],
        )
    )
    for class_name, weight
    in raw_class_weights.items()
}

full_supervised_training_pool[
    "class_weight"
] = (
    full_supervised_training_pool[
        "training_target"
    ]
    .map(class_weights)
    .astype(float)
)

full_supervised_training_pool[
    "label_reliability_weight"
] = (
    full_supervised_training_pool[
        "weak_label_confidence"
    ]
    .astype(float)
)

provisional_low_rows = (
    full_supervised_training_pool[
        "training_label_source"
    ].eq("provisional_low")
)

full_supervised_training_pool.loc[
    provisional_low_rows,
    "label_reliability_weight",
] *= 0.60

full_supervised_training_pool[
    "parser_reliability_weight"
] = np.where(
    full_supervised_training_pool[
        "parser_mode"
    ].eq("structured"),
    1.00,
    0.80,
)

full_supervised_training_pool[
    "conflict_reliability_weight"
] = np.where(
    full_supervised_training_pool[
        "rule_conflict"
    ],
    0.75,
    1.00,
)

full_supervised_training_pool[
    "sample_weight"
] = (
    full_supervised_training_pool[
        "class_weight"
    ]
    * full_supervised_training_pool[
        "label_reliability_weight"
    ]
    * full_supervised_training_pool[
        "parser_reliability_weight"
    ]
    * full_supervised_training_pool[
        "conflict_reliability_weight"
    ]
)

full_supervised_training_pool[
    "sample_weight"
] = (
    full_supervised_training_pool[
        "sample_weight"
    ]
    .clip(
        lower=CLEANING_SETTINGS[
            "minimum_sample_weight"
        ],
        upper=CLEANING_SETTINGS[
            "maximum_sample_weight"
        ],
    )
)

class_weight_report = pd.DataFrame(
    [
        {
            "training_target": class_name,
            "records": int(
                class_counts[class_name]
            ),
            "class_weight": round(
                class_weights[class_name],
                4,
            ),
        }
        for class_name in sorted(
            class_counts.index
        )
    ]
)

display(class_weight_report)

display(
    full_supervised_training_pool
    .groupby("training_target")[
        "sample_weight"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max",
        ]
    )
    .reset_index()
)

,training_target,records,class_weight
0,benign,26107,0.2245
1,critical,358,10.0000
2,high,566,10.0000
3,low,2128,2.7537
4,medium,140,10.0000


,training_target,count,mean,median,min,max
0,benign,26107,0.206497,0.206497,0.206497,0.206497
1,critical,358,9.808659,9.800000,9.800000,9.900000
2,high,566,9.493640,9.500000,7.200000,9.600000
3,low,2128,1.255671,1.255671,1.255671,1.255671
4,medium,140,9.100000,9.100000,9.100000,9.100000


In [21]:
risky_training_records = (
    full_supervised_training_pool[
        full_supervised_training_pool[
            "training_target"
        ].isin(
            [
                "low",
                "medium",
                "high",
                "critical",
            ]
        )
    ]
    .copy()
)

benign_training_candidates = (
    full_supervised_training_pool[
        full_supervised_training_pool[
            "training_target"
        ].eq("benign")
    ]
    .copy()
)

benign_sampling_target = min(
    len(benign_training_candidates),
    int(
        math.ceil(
            len(risky_training_records)
            * CLEANING_SETTINGS[
                "benign_to_risky_ratio"
            ]
        )
    ),
)

sampled_benign_records = (
    stratified_round_robin_sample(
        dataframe=benign_training_candidates,
        target_size=benign_sampling_target,
        stratification_columns=[
            "repository",
            "configuration_type",
            "operation",
            "parser_mode",
        ],
        random_seed=(
            CLEANING_SETTINGS[
                "random_seed"
            ]
            + 100
        ),
    )
)

balanced_training_pool = pd.concat(
    [
        sampled_benign_records,
        risky_training_records,
    ],
    ignore_index=True,
)

balanced_training_pool = (
    balanced_training_pool
    .sample(
        frac=1,
        random_state=CLEANING_SETTINGS[
            "random_seed"
        ],
    )
    .reset_index(drop=True)
)

print(
    "Risky training records :",
    f"{len(risky_training_records):,}",
)

print(
    "Benign records selected:",
    f"{len(sampled_benign_records):,}",
)

print(
    "Balanced training pool :",
    f"{len(balanced_training_pool):,}",
)

display(
    balanced_training_pool[
        "training_target"
    ]
    .value_counts()
    .reindex(
        [
            "benign",
            "low",
            "medium",
            "high",
            "critical",
        ],
        fill_value=0,
    )
    .rename_axis("training_target")
    .reset_index(name="records")
)

Risky training records : 3,192
Benign records selected: 6,384
Balanced training pool : 9,576


,training_target,records
0,benign,6384
1,low,2128
2,medium,140
3,high,566
4,critical,358


In [22]:
def assign_temporal_commit_partition(
    dataframe,
    development_fraction,
):
    """
    Assign complete commits to either model_train or temporal_dev.

    The most recent fraction of commits in each training repository
    becomes temporal_dev.
    """

    partitioned_data = dataframe.copy()

    commit_table = (
        partitioned_data
        .groupby(
            [
                "repository",
                "commit_hash",
            ],
            as_index=False,
        )
        .agg(
            commit_author_datetime=(
                "commit_author_datetime",
                "min",
            )
        )
    )

    development_commit_keys = set()

    for repository_name, repository_commits in (
        commit_table.groupby("repository")
    ):
        repository_commits = (
            repository_commits
            .sort_values(
                [
                    "commit_author_datetime",
                    "commit_hash",
                ],
                na_position="first",
            )
            .reset_index(drop=True)
        )

        number_of_commits = len(
            repository_commits
        )

        if number_of_commits <= 1:
            continue

        development_count = max(
            1,
            int(
                math.ceil(
                    number_of_commits
                    * development_fraction
                )
            ),
        )

        development_count = min(
            development_count,
            number_of_commits - 1,
        )

        development_commits = (
            repository_commits
            .tail(development_count)
        )

        development_commit_keys.update(
            zip(
                development_commits[
                    "repository"
                ],
                development_commits[
                    "commit_hash"
                ],
            )
        )

    partitioned_data[
        "temporal_partition"
    ] = [
        (
            "temporal_dev"
            if (
                repository,
                commit_hash,
            ) in development_commit_keys
            else "model_train"
        )
        for repository, commit_hash
        in zip(
            partitioned_data["repository"],
            partitioned_data["commit_hash"],
        )
    ]

    return partitioned_data


balanced_training_pool = (
    assign_temporal_commit_partition(
        dataframe=balanced_training_pool,
        development_fraction=(
            CLEANING_SETTINGS[
                "temporal_development_fraction"
            ]
        ),
    )
)

model_training_data = (
    balanced_training_pool[
        balanced_training_pool[
            "temporal_partition"
        ].eq("model_train")
    ]
    .copy()
)

temporal_development_data = (
    balanced_training_pool[
        balanced_training_pool[
            "temporal_partition"
        ].eq("temporal_dev")
    ]
    .copy()
)

print(
    "Model-training records :",
    f"{len(model_training_data):,}",
)

print(
    "Temporal-dev records   :",
    f"{len(temporal_development_data):,}",
)

display(
    balanced_training_pool
    .groupby(
        [
            "repository",
            "temporal_partition",
        ],
        as_index=False,
    )
    .agg(
        records=("diff_id", "count"),
        commits=("commit_hash", "nunique"),
    )
)

Model-training records : 7,711
Temporal-dev records   : 1,865


,repository,temporal_partition,records,commits
0,kube_prometheus,model_train,2486,409
1,kube_prometheus,temporal_dev,668,103
2,microservices_demo,model_train,4047,514
3,microservices_demo,temporal_dev,959,129
4,terraform_aws_vpc,model_train,1178,135
5,terraform_aws_vpc,temporal_dev,238,34


In [23]:
temporal_label_distribution = (
    balanced_training_pool
    .groupby(
        [
            "temporal_partition",
            "training_target",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "records",
        }
    )
)

display(
    temporal_label_distribution.sort_values(
        [
            "temporal_partition",
            "records",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

,temporal_partition,training_target,records
0,model_train,benign,5099
3,model_train,low,1724
2,model_train,high,471
1,model_train,critical,294
4,model_train,medium,123
5,temporal_dev,benign,1285
8,temporal_dev,low,404
7,temporal_dev,high,95
6,temporal_dev,critical,64
9,temporal_dev,medium,17


In [24]:
validation_review_pool = (
    clean_manifest[
        clean_manifest[
            "dataset_split"
        ].eq("validation")
    ]
    .copy()
)

validation_review_pool[
    "allowed_usage"
] = "review_and_evaluation_only"

test_review_pool = (
    clean_manifest[
        clean_manifest[
            "dataset_split"
        ].eq("test")
    ]
    .copy()
)

test_review_pool[
    "allowed_usage"
] = "final_evaluation_only"

unlabeled_pool = (
    clean_manifest[
        clean_manifest[
            "weak_label"
        ].eq("unlabeled")
    ]
    .copy()
)

unlabeled_pool[
    "allowed_usage"
] = "unsupervised_or_future_annotation"

print(
    "Validation review records:",
    f"{len(validation_review_pool):,}",
)

print(
    "Test review records      :",
    f"{len(test_review_pool):,}",
)

print(
    "Unlabeled records        :",
    f"{len(unlabeled_pool):,}",
)

Validation review records: 7,628
Test review records      : 68,749
Unlabeled records        : 26,539


In [25]:
manual_annotation_queue_path = (
    EVALUATION_DIR
    / "manual_annotation_queue.csv"
)

clean_annotation_queue = pd.DataFrame()

if manual_annotation_queue_path.exists():
    manual_annotation_queue = pd.read_csv(
        manual_annotation_queue_path,
        low_memory=False,
    )

    clean_diff_ids = set(
        clean_manifest["diff_id"]
    )

    clean_annotation_queue = (
        manual_annotation_queue[
            manual_annotation_queue[
                "diff_id"
            ].isin(clean_diff_ids)
        ]
        .copy()
    )

    removed_annotation_rows = (
        len(manual_annotation_queue)
        - len(clean_annotation_queue)
    )

    print(
        "Original annotation queue:",
        f"{len(manual_annotation_queue):,}",
    )

    print(
        "Clean annotation queue   :",
        f"{len(clean_annotation_queue):,}",
    )

    print(
        "Queue rows removed       :",
        f"{removed_annotation_rows:,}",
    )

else:
    print(
        "Manual annotation queue not found."
    )

Original annotation queue: 1,800
Clean annotation queue   : 1,383
Queue rows removed       : 417


In [26]:
temporary_columns = [
    "_structured_priority",
    "_no_conflict_priority",
    "_split_priority",
]

for dataframe in [
    clean_manifest,
    rejected_invalid_rows,
    removed_duplicate_diff_ids,
    removed_within_split_duplicates,
    removed_cross_split_rows,
    full_supervised_training_pool,
    balanced_training_pool,
    model_training_data,
    temporal_development_data,
    validation_review_pool,
    test_review_pool,
    unlabeled_pool,
]:
    dataframe.drop(
        columns=temporary_columns,
        errors="ignore",
        inplace=True,
    )

print("Temporary processing columns removed.")

Temporary processing columns removed.


In [27]:
OUTPUT_PATHS = {
    "clean_manifest": (
        CLEAN_DATA_DIR
        / "clean_manifest_all_splits.csv.gz"
    ),

    "rejected_invalid_rows": (
        CLEAN_DATA_DIR
        / "rejected_invalid_rows.csv.gz"
    ),

    "removed_duplicate_diff_ids": (
        CLEAN_DATA_DIR
        / "removed_duplicate_diff_ids.csv.gz"
    ),

    "removed_within_split_duplicates": (
        CLEAN_DATA_DIR
        / "removed_within_split_duplicate_signatures.csv.gz"
    ),

    "removed_cross_split_rows": (
        CLEAN_DATA_DIR
        / "removed_cross_split_leakage_rows.csv.gz"
    ),

    "full_supervised_training_pool": (
        CLEAN_DATA_DIR
        / "train_supervised_full.csv.gz"
    ),

    "balanced_training_pool": (
        CLEAN_DATA_DIR
        / "train_supervised_balanced.csv.gz"
    ),

    "model_training_data": (
        CLEAN_DATA_DIR
        / "train_temporal_model_train.csv.gz"
    ),

    "temporal_development_data": (
        CLEAN_DATA_DIR
        / "train_temporal_development.csv.gz"
    ),

    "validation_review_pool": (
        CLEAN_DATA_DIR
        / "validation_review_only.csv.gz"
    ),

    "test_review_pool": (
        CLEAN_DATA_DIR
        / "test_final_evaluation_only.csv.gz"
    ),

    "unlabeled_pool": (
        CLEAN_DATA_DIR
        / "unlabeled_annotation_pool.csv.gz"
    ),
}

dataframes_to_save = {
    "clean_manifest": clean_manifest,
    "rejected_invalid_rows":
        rejected_invalid_rows,
    "removed_duplicate_diff_ids":
        removed_duplicate_diff_ids,
    "removed_within_split_duplicates":
        removed_within_split_duplicates,
    "removed_cross_split_rows":
        removed_cross_split_rows,
    "full_supervised_training_pool":
        full_supervised_training_pool,
    "balanced_training_pool":
        balanced_training_pool,
    "model_training_data":
        model_training_data,
    "temporal_development_data":
        temporal_development_data,
    "validation_review_pool":
        validation_review_pool,
    "test_review_pool":
        test_review_pool,
    "unlabeled_pool":
        unlabeled_pool,
}

for output_name, dataframe in (
    dataframes_to_save.items()
):
    output_path = OUTPUT_PATHS[
        output_name
    ]

    dataframe.to_csv(
        output_path,
        index=False,
        compression="gzip",
    )

    print(
        f"{output_name:<35}",
        f"{len(dataframe):>9,} rows",
        "->",
        output_path.name,
    )

if not clean_annotation_queue.empty:
    clean_annotation_queue_path = (
        EVALUATION_DIR
        / "manual_annotation_queue_cleaned.csv"
    )

    clean_annotation_queue.to_csv(
        clean_annotation_queue_path,
        index=False,
    )

    print(
        "\nClean annotation queue saved:",
        clean_annotation_queue_path,
    )

clean_manifest                        219,868 rows -> clean_manifest_all_splits.csv.gz
rejected_invalid_rows                   8,841 rows -> rejected_invalid_rows.csv.gz
removed_duplicate_diff_ids                  0 rows -> removed_duplicate_diff_ids.csv.gz
removed_within_split_duplicates       129,838 rows -> removed_within_split_duplicate_signatures.csv.gz
removed_cross_split_rows                1,191 rows -> removed_cross_split_leakage_rows.csv.gz
full_supervised_training_pool          29,299 rows -> train_supervised_full.csv.gz
balanced_training_pool                  9,576 rows -> train_supervised_balanced.csv.gz
model_training_data                     7,711 rows -> train_temporal_model_train.csv.gz
temporal_development_data               1,865 rows -> train_temporal_development.csv.gz
validation_review_pool                  7,628 rows -> validation_review_only.csv.gz
test_review_pool                       68,749 rows -> test_final_evaluation_only.csv.gz
unlabeled_pool             

In [28]:
post_clean_signature_split_counts = (
    clean_manifest
    .groupby(
        "change_signature_sha256"
    )["dataset_split"]
    .nunique()
)

remaining_cross_split_signatures = int(
    (
        post_clean_signature_split_counts
        > 1
    ).sum()
)

print(
    "Remaining cross-split duplicate signatures:",
    remaining_cross_split_signatures,
)

if remaining_cross_split_signatures != 0:
    raise ValueError(
        "Cross-split signature leakage remains."
    )

print("Cross-split signature leakage check: PASSED")

Remaining cross-split duplicate signatures: 0
Cross-split signature leakage check: PASSED


In [29]:
model_train_commit_keys = set(
    zip(
        model_training_data["repository"],
        model_training_data["commit_hash"],
    )
)

temporal_dev_commit_keys = set(
    zip(
        temporal_development_data[
            "repository"
        ],
        temporal_development_data[
            "commit_hash"
        ],
    )
)

overlapping_temporal_commits = (
    model_train_commit_keys
    & temporal_dev_commit_keys
)

print(
    "Model-train commits :",
    f"{len(model_train_commit_keys):,}",
)

print(
    "Temporal-dev commits:",
    f"{len(temporal_dev_commit_keys):,}",
)

print(
    "Overlapping commits :",
    f"{len(overlapping_temporal_commits):,}",
)

if overlapping_temporal_commits:
    raise ValueError(
        "Commit leakage exists between model_train "
        "and temporal_dev."
    )

print("Temporal commit separation check: PASSED")

Model-train commits : 1,058
Temporal-dev commits: 266
Overlapping commits : 0
Temporal commit separation check: PASSED


In [30]:
model_train_commit_keys = set(
    zip(
        model_training_data["repository"],
        model_training_data["commit_hash"],
    )
)

temporal_dev_commit_keys = set(
    zip(
        temporal_development_data[
            "repository"
        ],
        temporal_development_data[
            "commit_hash"
        ],
    )
)

overlapping_temporal_commits = (
    model_train_commit_keys
    & temporal_dev_commit_keys
)

print(
    "Model-train commits :",
    f"{len(model_train_commit_keys):,}",
)

print(
    "Temporal-dev commits:",
    f"{len(temporal_dev_commit_keys):,}",
)

print(
    "Overlapping commits :",
    f"{len(overlapping_temporal_commits):,}",
)

if overlapping_temporal_commits:
    raise ValueError(
        "Commit leakage exists between model_train "
        "and temporal_dev."
    )

print("Temporal commit separation check: PASSED")

Model-train commits : 1,058
Temporal-dev commits: 266
Overlapping commits : 0
Temporal commit separation check: PASSED


In [32]:
training_repositories = set(
    balanced_training_pool[
        "repository"
    ].unique()
)

validation_repositories = set(
    validation_review_pool[
        "repository"
    ].unique()
)

test_repositories = set(
    test_review_pool[
        "repository"
    ].unique()
)

required_training_classes = {
    "benign",
    "low",
    "medium",
    "high",
    "critical",
}

available_training_classes = set(
    balanced_training_pool[
        "training_target"
    ].unique()
)

integrity_checks = {
    "Clean manifest is not empty":
        not clean_manifest.empty,

    "Diff IDs are unique":
        clean_manifest[
            "diff_id"
        ].is_unique,

    "Signatures are unique inside each split":
        ~clean_manifest.duplicated(
            subset=[
                "dataset_split",
                "change_signature_sha256",
            ]
        ).any(),

    "No cross-split signatures remain":
        remaining_cross_split_signatures == 0,

    "Each repository belongs to one split":
        repository_split_integrity[
            "unique_splits"
        ].max() == 1,

    "Training pool contains only train repositories":
        balanced_training_pool[
            "dataset_split"
        ].eq("train").all(),

    "Validation repositories are excluded from training":
        training_repositories.isdisjoint(
            validation_repositories
        ),

    "Test repositories are excluded from training":
        training_repositories.isdisjoint(
            test_repositories
        ),

    "Training pool contains no unlabeled targets":
        ~balanced_training_pool[
            "training_target"
        ].eq("unlabeled").any(),

    "All required training classes exist":
        required_training_classes.issubset(
            available_training_classes
        ),

    "Training sample weights are valid":
        balanced_training_pool[
            "sample_weight"
        ].between(
            CLEANING_SETTINGS[
                "minimum_sample_weight"
            ],
            CLEANING_SETTINGS[
                "maximum_sample_weight"
            ],
            inclusive="both",
        ).all(),

    "Model-train and temporal-dev commits are disjoint":
        len(
            overlapping_temporal_commits
        ) == 0,

    "Model-training partition is not empty":
        not model_training_data.empty,

    "Temporal-development partition is not empty":
        not temporal_development_data.empty,

    "Validation review pool is not empty":
        not validation_review_pool.empty,

    "Final test pool is not empty":
        not test_review_pool.empty,
}

print("Dataset integrity checks:\n")

for check_name, passed in (
    integrity_checks.items()
):
    print(
        f"{'PASSED' if passed else 'FAILED':<8}"
        f" | {check_name}"
    )

failed_integrity_checks = [
    check_name
    for check_name, passed
    in integrity_checks.items()
    if not bool(passed)
]

print("\n" + "=" * 72)

if failed_integrity_checks:
    print(
        "FAILED INTEGRITY CHECKS:",
        len(failed_integrity_checks),
    )

    for index, check_name in enumerate(
        failed_integrity_checks,
        start=1,
    ):
        print(f"{index}. {check_name}")
else:
    print("ALL DATASET INTEGRITY CHECKS PASSED")

print("=" * 72)

Dataset integrity checks:

PASSED   | Clean manifest is not empty
PASSED   | Diff IDs are unique
PASSED   | Signatures are unique inside each split
PASSED   | No cross-split signatures remain
PASSED   | Each repository belongs to one split
PASSED   | Training pool contains only train repositories
PASSED   | Validation repositories are excluded from training
PASSED   | Test repositories are excluded from training
PASSED   | Training pool contains no unlabeled targets
PASSED   | All required training classes exist
PASSED   | Training sample weights are valid
PASSED   | Model-train and temporal-dev commits are disjoint
PASSED   | Model-training partition is not empty
PASSED   | Temporal-development partition is not empty
PASSED   | Validation review pool is not empty
PASSED   | Final test pool is not empty

ALL DATASET INTEGRITY CHECKS PASSED


In [33]:
# Reconstruct failed checks safely before creating the summary

if "integrity_checks" not in globals():
    raise RuntimeError(
        "The variable 'integrity_checks' does not exist. "
        "Run Cell 31 — Main integrity checks before this cell."
    )

failed_integrity_checks = [
    check_name
    for check_name, passed
    in integrity_checks.items()
    if not bool(passed)
]

output_summary = {
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "input_records": int(
        len(manifest)
    ),

    "invalid_records_removed": int(
        len(rejected_invalid_rows)
    ),

    "duplicate_diff_ids_removed": int(
        len(removed_duplicate_diff_ids)
    ),

    "within_split_duplicate_signatures_removed": int(
        len(removed_within_split_duplicates)
    ),

    "cross_split_leakage_rows_removed": int(
        len(removed_cross_split_rows)
    ),

    "clean_records": int(
        len(clean_manifest)
    ),

    "full_supervised_training_records": int(
        len(full_supervised_training_pool)
    ),

    "balanced_training_records": int(
        len(balanced_training_pool)
    ),

    "model_training_records": int(
        len(model_training_data)
    ),

    "temporal_development_records": int(
        len(temporal_development_data)
    ),

    "validation_review_records": int(
        len(validation_review_pool)
    ),

    "test_final_evaluation_records": int(
        len(test_review_pool)
    ),

    "unlabeled_pool_records": int(
        len(unlabeled_pool)
    ),

    "provisional_low_records": int(
        len(provisional_low_sample)
    ),

    "training_class_distribution": {
        str(label): int(count)
        for label, count
        in balanced_training_pool[
            "training_target"
        ].value_counts().items()
    },

    "remaining_cross_split_signatures": int(
        remaining_cross_split_signatures
    ),

    "failed_integrity_checks": (
        failed_integrity_checks
    ),

    "all_integrity_checks_passed": (
        len(failed_integrity_checks) == 0
    ),

    "output_paths": {
        name: str(path)
        for name, path
        in OUTPUT_PATHS.items()
    },
}

output_summary_path = (
    CLEAN_DATA_DIR
    / "dataset_cleaning_summary.json"
)

with output_summary_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        output_summary,
        file,
        indent=2,
    )

print(
    "Saved cleaning summary:",
    output_summary_path,
)

print(
    "Failed integrity checks:",
    len(failed_integrity_checks),
)

if failed_integrity_checks:
    for index, check_name in enumerate(
        failed_integrity_checks,
        start=1,
    ):
        print(f"{index}. {check_name}")
else:
    print("All dataset integrity checks passed.")

Saved cleaning summary: C:\Users\Lenovo\Desktop\DriftGuard\data\clean\dataset_cleaning_summary.json
Failed integrity checks: 0
All dataset integrity checks passed.


In [34]:
print("=" * 72)
print("NOTEBOOK 05 — REQUIRED RESULTS")
print("=" * 72)

print("\n1. CLEANING COUNTS")
print("-" * 72)

print(
    "Input records                         :",
    f"{len(manifest):,}",
)

print(
    "Invalid records removed               :",
    f"{len(rejected_invalid_rows):,}",
)

print(
    "Duplicate Diff IDs removed            :",
    f"{len(removed_duplicate_diff_ids):,}",
)

print(
    "Within-split duplicate signatures     :",
    f"{len(removed_within_split_duplicates):,}",
)

print(
    "Cross-split leakage rows removed       :",
    f"{len(removed_cross_split_rows):,}",
)

print(
    "Final clean records                   :",
    f"{len(clean_manifest):,}",
)

print("\n2. TRAINING DATASETS")
print("-" * 72)

print(
    "High-confidence core records          :",
    f"{len(core_training_pool):,}",
)

print(
    "Provisional low-risk records          :",
    f"{len(provisional_low_sample):,}",
)

print(
    "Full supervised training pool         :",
    f"{len(full_supervised_training_pool):,}",
)

print(
    "Balanced supervised training pool     :",
    f"{len(balanced_training_pool):,}",
)

print(
    "Temporal model-training records       :",
    f"{len(model_training_data):,}",
)

print(
    "Temporal development records          :",
    f"{len(temporal_development_data):,}",
)

print("\nBalanced class distribution:")

display(
    balanced_training_pool[
        "training_target"
    ]
    .value_counts()
    .reindex(
        [
            "benign",
            "low",
            "medium",
            "high",
            "critical",
        ],
        fill_value=0,
    )
    .rename_axis("training_target")
    .reset_index(name="records")
)

print("\n3. ISOLATED EVALUATION DATA")
print("-" * 72)

print(
    "Validation review-only records        :",
    f"{len(validation_review_pool):,}",
)

print(
    "Final test evaluation-only records    :",
    f"{len(test_review_pool):,}",
)

print(
    "Unlabeled annotation records          :",
    f"{len(unlabeled_pool):,}",
)

print("\n4. LEAKAGE CHECK")
print("-" * 72)

print(
    "Remaining cross-split signatures      :",
    f"{remaining_cross_split_signatures:,}",
)

print(
    "Temporal commit overlap               :",
    f"{len(overlapping_temporal_commits):,}",
)

print("\n5. FAILED INTEGRITY CHECKS")
print("-" * 72)

if failed_integrity_checks:
    for index, check_name in enumerate(
        failed_integrity_checks,
        start=1,
    ):
        print(f"{index}. {check_name}")
else:
    print("No failed integrity checks.")
    print("All cleaning and leakage checks passed.")

print("\n" + "=" * 72)
print("REQUIRED RESULTS GENERATED")
print("=" * 72)

NOTEBOOK 05 — REQUIRED RESULTS

1. CLEANING COUNTS
------------------------------------------------------------------------
Input records                         : 359,738
Invalid records removed               : 8,841
Duplicate Diff IDs removed            : 0
Within-split duplicate signatures     : 129,838
Cross-split leakage rows removed       : 1,191
Final clean records                   : 219,868

2. TRAINING DATASETS
------------------------------------------------------------------------
High-confidence core records          : 27,171
Provisional low-risk records          : 2,128
Full supervised training pool         : 29,299
Balanced supervised training pool     : 9,576
Temporal model-training records       : 7,711
Temporal development records          : 1,865

Balanced class distribution:


,training_target,records
0,benign,6384
1,low,2128
2,medium,140
3,high,566
4,critical,358



3. ISOLATED EVALUATION DATA
------------------------------------------------------------------------
Validation review-only records        : 7,628
Final test evaluation-only records    : 68,749
Unlabeled annotation records          : 26,539

4. LEAKAGE CHECK
------------------------------------------------------------------------
Remaining cross-split signatures      : 0
Temporal commit overlap               : 0

5. FAILED INTEGRITY CHECKS
------------------------------------------------------------------------
No failed integrity checks.
All cleaning and leakage checks passed.

REQUIRED RESULTS GENERATED


In [35]:
print("=" * 72)
print("NOTEBOOK 05 COMPLETED")
print("=" * 72)

print(
    "Original weak-label records     :",
    f"{len(manifest):,}",
)

print(
    "Final leakage-safe records      :",
    f"{len(clean_manifest):,}",
)

print(
    "Balanced training records       :",
    f"{len(balanced_training_pool):,}",
)

print(
    "Model-training records          :",
    f"{len(model_training_data):,}",
)

print(
    "Temporal development records    :",
    f"{len(temporal_development_data):,}",
)

print(
    "Validation review-only records  :",
    f"{len(validation_review_pool):,}",
)

print(
    "Final test-only records         :",
    f"{len(test_review_pool):,}",
)

print(
    "Cross-split signatures remaining:",
    f"{remaining_cross_split_signatures:,}",
)

print("\nImportant:")
print("- Test repositories were never added to the training pool.")
print("- Validation repositories remain review and evaluation only.")
print("- Duplicate signatures were retained according to test-first priority.")
print("- Low-risk labels remain provisional and receive reduced weights.")
print("- Temporal partitions contain disjoint commits.")
print("- Final validation/test metrics still require human-reviewed labels.")

print("\nNext notebook:")
print("06_exploratory_data_analysis.ipynb")

NOTEBOOK 05 COMPLETED
Original weak-label records     : 359,738
Final leakage-safe records      : 219,868
Balanced training records       : 9,576
Model-training records          : 7,711
Temporal development records    : 1,865
Validation review-only records  : 7,628
Final test-only records         : 68,749
Cross-split signatures remaining: 0

Important:
- Test repositories were never added to the training pool.
- Validation repositories remain review and evaluation only.
- Duplicate signatures were retained according to test-first priority.
- Low-risk labels remain provisional and receive reduced weights.
- Temporal partitions contain disjoint commits.
- Final validation/test metrics still require human-reviewed labels.

Next notebook:
06_exploratory_data_analysis.ipynb
